# fiXAIt 0.8.0 — Benchmark Demo

This notebook runs the unified global and local fiXAIt workflow by selecting one of the five benchmark datasets in the repository. `OPTIMIZE_FAITHFULNESS` and `OPTIMIZE_LOCAL_FAITHFULNESS` independently select the single final global and local output modes.


## 1. Installation

Install the package once in the VS Code WSL terminal:

```bash
python3 -m pip install -e "/mnt/d/calismalar/fixait[optimizer]"
```

If you will not use the optimizer, you can remove the `[optimizer]` extra.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

try:
    from IPython.display import display
except ImportError:
    display = print


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/mnt/d/calismalar/fixait')]
    for candidate in candidates:
        if (candidate / 'benchmarks' / 'data').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        'The fiXAIt project directory could not be found. If necessary, change the Path("/mnt/d/calismalar/fixait") path to match your own location.'
    )


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'benchmarks' / 'data'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import fixait
from fixait import FiXAIt, FiXAItConfig

if tuple(int(part) for part in fixait.__version__.split('.')[:3]) < (0, 8, 0):
    raise RuntimeError(
        'This demo requires fiXAIt >= 0.8.0. Run python3 -m pip install -e "/mnt/d/calismalar/fixait[optimizer]" in the WSL terminal.'
    )

print('fiXAIt version:', fixait.__version__)
print('Project:', PROJECT_ROOT)
print('Data:', DATA_ROOT)


## 2. Experiment Settings

For `DATASET_NAME`, you can select `adult`, `boston_housing`, `compas`, `german_credit`, or `student`. The main results include only the FEI/FVI values for the modes selected through the global and local optimization flags.


In [ ]:
DATASET_SPECS = {
    'adult': ('adult.csv', 'income_label'),
    'boston_housing': ('boston_housing.csv', 'price_band'),
    'compas': ('compas.csv', 'risk_level'),
    'german_credit': ('german_credit.csv', 'creditability'),
    'student': ('student.csv', 'grade'),
}

DATASET_NAME = 'student'
OPTIMIZE_FAITHFULNESS = False
OPTIMIZE_LOCAL_FAITHFULNESS = False
MAX_ROWS = 600
RANDOM_STATE = 42

print('Dataset:', DATASET_NAME)
print('Global faithfulness optimization:', OPTIMIZE_FAITHFULNESS)
print('Local faithfulness optimization:', OPTIMIZE_LOCAL_FAITHFULNESS)


In [ ]:
if DATASET_NAME not in DATASET_SPECS:
    raise ValueError(f'Unknown dataset: {DATASET_NAME}')

file_name, target_column = DATASET_SPECS[DATASET_NAME]
data = pd.read_csv(DATA_ROOT / file_name)
if len(data) > MAX_ROWS:
    data = data.sample(MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
data = data.rename(columns={'class': target_column})

X = data.drop(columns=target_column)
y = data[target_column]

print('Shape:', data.shape)
print('Target column:', target_column)
print('Classes:', sorted(pd.unique(y).tolist()))
display(data.head())


## 3. Model and fiXAIt Configuration

To keep the demo runtime reasonable, the numbers of global and local faithfulness perturbations and optimizer steps have been set to low values. For final experiments, use at least `30`, `30`, and `500`, respectively.


In [ ]:
model = RandomForestClassifier(
    n_estimators=20,
    max_depth=4,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

config = FiXAItConfig(
    group_size=min(7, X.shape[1]),
    optimize_faithfulness=OPTIMIZE_FAITHFULNESS,
    optimize_local_faithfulness=OPTIMIZE_LOCAL_FAITHFULNESS,
    fei_threshold_pct=3.0,
    faithfulness_runs_per_feature=5,
    local_faithfulness_runs_per_feature=5,
    local_faithfulness_calibration_runs_per_feature=5,
    faithfulness_optimizer_steps=100,
    faithfulness_reg_lambda=0.10,
    faithfulness_accept_only_if_improved=True,
    faithfulness_min_improvement=0.01,
    faithfulness_max_weight_change_pct=20.0,
    local_faithfulness_optimizer_steps=100,
    local_faithfulness_reg_lambda=0.10,
    local_faithfulness_accept_only_if_improved=True,
    local_faithfulness_min_improvement=0.01,
    local_faithfulness_max_weight_change_pct=20.0,
    fidelity_top_k=7,
    random_state=RANDOM_STATE,
    n_jobs=1,
    model_n_jobs=1,
)

explainer = FiXAIt(model, config=config).fit(
    data,
    target_column=target_column,
)

print('The model and fiXAIt core are ready.')


## 4. Global Explanation

FEI and FVI use the same final feature set. As in the reference notebook, global FVI is returned with three decimal places without L1 normalization.


In [ ]:
global_result = explainer.explain_global()
optimization = global_result.metadata['optimization']

global_table = pd.DataFrame({
    'FEI': pd.Series(global_result.global_fei),
    'FVI': pd.Series(global_result.global_fvi),
}).sort_values('FEI', ascending=False)

print('Optimization requested:', optimization['requested'])
print('Optimization accepted:', optimization['accepted'])
print('Reason for acceptance/rejection:', optimization['reason'])
if optimization['requested']:
    print('Validation faithfulness change:', optimization['validation_faithfulness_improvement'])
    print('Mean weight change (%):', optimization['mean_weight_change_pct'])
    print('Maximum weight change (%):', optimization['max_weight_change_pct'])
print('Selected features:', global_result.selected_features)
print('Dropped features:', global_result.dropped_features)
display(global_table)

global_metrics = pd.Series({
    'Self-consistency': global_result.global_sc.overall,
    'Faithfulness': global_result.faithfulness,
    'Fidelity': global_result.fidelity,
    'Selected-feature accuracy': global_result.selected_feature_accuracy,
})
display(global_metrics.to_frame('Value'))


In [ ]:
faithfulness_details = pd.DataFrame({
    'Permutation impact': pd.Series(global_result.global_faithfulness.drop_impacts),
})
print('Faithfulness details:')
display(faithfulness_details.sort_values('Permutation impact', ascending=False))

print('Fidelity features:', global_result.global_fidelity.selected_features)
print('Surrogate tree depth:', global_result.global_fidelity.best_max_depth)


## 5. Local Explanation

The cell below explains the first row with respect to the class predicted by the model for that row. Local FEI applies the `fei_threshold_pct` value to the absolute contribution; the direction of negative contributions is preserved, and the same feature set is reflected in local FVI. FEI–FVI agreement and independent perturbation-based local faithfulness are reported separately.


In [ ]:
ROW_INDEX = 0
local_result = explainer.explain_local(X.iloc[ROW_INDEX])

local_table = pd.DataFrame({
    'Local FEI': pd.Series(local_result.local_fei),
    'Local FVI': pd.Series(local_result.local_fvi),
}).sort_values('Local FEI', key=lambda values: values.abs(), ascending=False)

print('Explained row:', ROW_INDEX)
print('Explained class:', local_result.target_class)
print('Combination strategy:', local_result.metadata['combination_strategy'])
print('Number of combinations:', local_result.metadata['n_combinations'])
print('Number of surrogate rows:', local_result.metadata['n_surrogate_rows'])
print('Empty coalition included:', local_result.metadata['empty_coalition_included'])
print('Ridge CV strategy:', local_result.metadata['ridge_cv_strategy'])
print('Ridge CV folds:', local_result.metadata['ridge_cv_splits'])
print('Selected local features:', local_result.selected_features)
print('Dropped local features:', local_result.dropped_features)
print('Local SC:', local_result.local_sc.overall)
print('Local FEI–FVI agreement (Spearman):', local_result.fei_fvi_agreement_spearman)
print('Local faithfulness (Spearman):', local_result.local_faithfulness_spearman)
print('Local optimization applied:', local_result.optimization_applied)
print('Local optimization metadata:', local_result.metadata['optimization'])
print('Local faithfulness informative:', local_result.metadata['local_faithfulness_informative'])
print('Local faithfulness perturbations per feature:', local_result.metadata['local_faithfulness_runs_per_feature'])
print('Local fidelity R²:', local_result.fidelity_r2)
display(pd.DataFrame({
    'Independent perturbation impact': pd.Series(local_result.metadata['local_faithfulness_impacts']),
}).sort_values('Independent perturbation impact', ascending=False))
display(local_table)


## 6. Optional: All Five Datasets

If you set `RUN_ALL_DATASETS=True`, the same selected optimization mode will run on all benchmark datasets using fast settings.


In [ ]:
RUN_ALL_DATASETS = False


def run_quick_benchmark(dataset_name):
    file_name, label_name = DATASET_SPECS[dataset_name]
    frame = pd.read_csv(DATA_ROOT / file_name)
    if len(frame) > 300:
        frame = frame.sample(300, random_state=RANDOM_STATE).reset_index(drop=True)
    frame = frame.rename(columns={'class': label_name})
    feature_count = frame.shape[1] - 1

    quick_explainer = FiXAIt(
        RandomForestClassifier(
            n_estimators=8,
            max_depth=3,
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        config=FiXAItConfig(
            group_size=min(7, feature_count),
            optimize_faithfulness=OPTIMIZE_FAITHFULNESS,
            optimize_local_faithfulness=OPTIMIZE_LOCAL_FAITHFULNESS,
            faithfulness_runs_per_feature=2,
            local_faithfulness_runs_per_feature=3,
            local_faithfulness_calibration_runs_per_feature=3,
            faithfulness_optimizer_steps=50,
            local_faithfulness_optimizer_steps=50,
            n_jobs=1,
            model_n_jobs=1,
            random_state=RANDOM_STATE,
        ),
    ).fit(frame, target_column=label_name)

    result = quick_explainer.explain_global()
    local_result = quick_explainer.explain_local(frame.drop(columns=label_name).iloc[0])
    return {
        'dataset': dataset_name,
        'optimization_requested': result.metadata['optimization']['requested'],
        'optimization_accepted': result.metadata['optimization']['accepted'],
        'selected_count': len(result.selected_features),
        'selected_features': ', '.join(result.selected_features),
        'SC': result.global_sc.overall,
        'faithfulness': result.faithfulness,
        'fidelity': result.fidelity,
        'local_fei_fvi_agreement': local_result.fei_fvi_agreement_spearman,
        'local_faithfulness': local_result.local_faithfulness_spearman,
        'local_optimization_accepted': local_result.optimization_applied,
        'local_fidelity_r2': local_result.fidelity_r2,
    }


if RUN_ALL_DATASETS:
    benchmark_summary = pd.DataFrame(
        [run_quick_benchmark(name) for name in DATASET_SPECS]
    )
    display(benchmark_summary)
else:
    print('Set RUN_ALL_DATASETS=True to run all datasets.')
